In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import os
import pretty_midi

# 1. Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Load Data (Adjust path if needed)
csv_path = '../data/maestro-v3.0.0/maestro-v3.0.0.csv'
metadata = pd.read_csv(csv_path)
base_folder = '../data/maestro-v3.0.0/'

# Load the first MIDI file
first_midi_path = os.path.join(base_folder, metadata['midi_filename'].iloc[0])
midi_data = pretty_midi.PrettyMIDI(first_midi_path)
piano_roll = midi_data.get_piano_roll(fs=16)

# 3. Create Dataset function
def create_dataset(piano_roll, sequence_length=100):
    inputs, targets = [], []
    for i in range(piano_roll.shape[1] - sequence_length):
        inputs.append(piano_roll[:, i:i + sequence_length])
        targets.append(piano_roll[:, i + sequence_length])
    return np.array(inputs), np.array(targets)

# 4. Prepare Tensors
X, y = create_dataset(piano_roll[:, :500])
X_tensor = (torch.FloatTensor(X) > 0).float().transpose(1, 2).to(device)
y_tensor = (torch.FloatTensor(y) > 0).float().to(device)

print(f"Data ready on {device}!")
print(f"Input Shape: {X_tensor.shape}")

Data ready on cpu!
Input Shape: torch.Size([400, 100, 128])


In [2]:
class MusicTransformer(nn.Module):
    def __init__(self, n_features=128, n_head=8, n_layers=3, dropout=0.1):
        super(MusicTransformer, self).__init__()
        self.encoder = nn.Linear(n_features, 256)
        self.pos_encoder = nn.Parameter(torch.zeros(1, 100, 256))

        # Transformer with causal (look-ahead) mask
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=256, nhead=n_head, dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=n_layers)

        self.decoder = nn.Linear(256, 128)
        # NO sigmoid here — output raw logits

    def generate_causal_mask(self, seq_len, device):
        # Upper-triangular mask: each position can only attend to itself and PAST positions
        # This is the critical fix — without this, the model cheats by looking at future tokens
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask

    def forward(self, x):
        seq_len = x.size(1)
        causal_mask = self.generate_causal_mask(seq_len, x.device)

        x = self.encoder(x) + self.pos_encoder[:, :seq_len, :]
        # Pass the causal mask so the model cannot see future tokens
        x = self.transformer_encoder(x, mask=causal_mask)
        x = self.decoder(x[:, -1, :])
        return x  # Raw logits

model_trans = MusicTransformer().to(device)
print("Transformer Model Built with Causal Mask!")

Transformer Model Built with Causal Mask!


In [ ]:
import math

# Use BCEWithLogitsLoss — consistent with Task 1 fix (no double sigmoid)
num_positive = y_tensor.sum().item()
num_negative = y_tensor.numel() - num_positive
pos_weight_value = num_negative / (num_positive + 1e-6)
pos_weight = torch.tensor([pos_weight_value]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model_trans.parameters(), lr=0.0001)

num_epochs = 50
loss_history = []
perplexity_history = []

print("Training Transformer with Causal Mask...")

for epoch in range(num_epochs):
    model_trans.train()
    optimizer.zero_grad()

    logits = model_trans(X_tensor)         # Raw logits
    loss = criterion(logits, y_tensor)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_trans.parameters(), max_norm=1.0)
    optimizer.step()

    # Perplexity = exp(average cross-entropy loss per token)
    # We use the raw BCE loss (without pos_weight) for a clean perplexity value
    with torch.no_grad():
        raw_criterion = nn.BCEWithLogitsLoss()
        raw_loss = raw_criterion(logits, y_tensor)
        perplexity = math.exp(raw_loss.item())

    loss_history.append(loss.item())
    perplexity_history.append(perplexity)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}] | Loss: {loss.item():.4f} | Perplexity: {perplexity:.2f}')

print("\nTransformer Training Complete!")

# Plot loss and perplexity curves (required deliverable)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(loss_history)
axes[0].set_title('Task 3: Transformer Loss Over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (weighted BCE)')
axes[0].grid(True)

axes[1].plot(perplexity_history, color='orange')
axes[1].set_title('Task 3: Perplexity Over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Perplexity')
axes[1].grid(True)

plt.tight_layout()
plt.savefig('Task 3_Transformer-based Music Generator.png', dpi=150)
plt.show()
print("Loss and perplexity curves saved!")

# Final perplexity report
print(f"\n=== PERPLEXITY EVALUATION REPORT ===")
print(f"Final Perplexity: {perplexity_history[-1]:.2f}")
print(f"Best Perplexity:  {min(perplexity_history):.2f} (epoch {perplexity_history.index(min(perplexity_history))+1})")

Training Transformer with Causal Mask...


In [ ]:
model_trans.eval()
generated_music_trans = []
current_sequence = X_tensor[0:1]

print("Transformer is composing...")

with torch.no_grad():
    for i in range(500):
        logits = model_trans(current_sequence)
        # Apply sigmoid manually since model outputs raw logits
        prediction = torch.sigmoid(logits)
        generated_music_trans.append(prediction.cpu().numpy())

        new_step = prediction.unsqueeze(1)
        current_sequence = torch.cat((current_sequence[:, 1:, :], new_step), dim=1)

gen_pianoroll_trans = np.array(generated_music_trans).squeeze().T

# Visualize
plt.figure(figsize=(12, 6))
plt.imshow(gen_pianoroll_trans[:, :200], cmap='magma', aspect='auto', origin='lower')
plt.title('Task 3: Transformer Generated Music')
plt.xlabel('Time Steps')
plt.ylabel('MIDI Pitch')
plt.tight_layout()
plt.show()

print("Generation complete!")

In [ ]:
import numpy as np
import pretty_midi
import torch

# 1. Define Helper Functions (Internal to this notebook)
def get_pitch_histogram_final(pr, thresh):
    histogram = np.sum(pr > thresh, axis=1)
    if np.sum(histogram) > 0:
        histogram = histogram / np.sum(histogram)
    return histogram

def calculate_similarity_final(orig_hist, gen_hist):
    return np.minimum(orig_hist, gen_hist).sum()

def export_to_midi_final(pianoroll, filename, thresh=0.01):
    pm = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0)
    for note_num in range(128):
        notes_active = pianoroll[note_num, :] > thresh
        changes = np.diff(notes_active.astype(int), prepend=0, append=0)
        starts = np.where(changes == 1)[0]
        ends = np.where(changes == -1)[0]
        for s, e in zip(starts, ends):
            note = pretty_midi.Note(velocity=100, pitch=note_num, start=s/16, end=e/16)
            piano.notes.append(note)
    pm.instruments.append(piano)
    pm.write(filename)
    print(f"Successfully exported: {filename}")

# 2. Get Ground Truth (from your X_tensor)
original_pr = X_tensor[0].cpu().numpy().T
orig_hist = get_pitch_histogram_final(original_pr, 0.5)

# 3. Run Evaluation on Transformer Output
# Note: Ensure you run your Transformer Generation cell first so 'gen_pianoroll_trans' exists
trans_hist = get_pitch_histogram_final(gen_pianoroll_trans, 0.01)
trans_sim = calculate_similarity_final(orig_hist, trans_hist)

print(f"--- Transformer Evaluation ---")
print(f"Transformer Pitch Similarity: {trans_sim:.2%}")

# 4. Export the MIDI
export_to_midi_final(gen_pianoroll_trans, 'Transformer_Generated_Sample.mid')

In [7]:
import torch
import numpy as np
import pretty_midi

def export_to_midi(pianoroll, filename, thresh=0.001):
    pm = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0)
    note_count = 0
    for note_num in range(128):
        notes_active = pianoroll[note_num, :] > thresh
        changes = np.diff(notes_active.astype(int), prepend=0, append=0)
        starts = np.where(changes ==  1)[0]
        ends   = np.where(changes == -1)[0]
        for s, e in zip(starts, ends):
            note = pretty_midi.Note(velocity=80, pitch=note_num, start=s/16, end=e/16)
            piano.notes.append(note)
            note_count += 1
    pm.instruments.append(piano)
    pm.write(filename)
    print(f"Exported: {filename}  ({note_count} notes)")

# ── Generate 10 samples from different seed windows ───────────────────────
model_trans.eval()

for sample_idx in range(10):
    seed_idx = sample_idx * 5  # use different parts of the data as seeds
    current_sequence = X_tensor[seed_idx:seed_idx+1]
    generated = []

    with torch.no_grad():
        for _ in range(500):
            logits = model_trans(current_sequence)
            prediction = torch.sigmoid(logits)
            generated.append(prediction.cpu().numpy())
            new_step = prediction.unsqueeze(1)
            current_sequence = torch.cat((current_sequence[:, 1:, :], new_step), dim=1)

    pr = np.array(generated).squeeze().T  # shape (128, 500)
    above = (pr > 0.001).sum()
    print(f"Sample {sample_idx+1}: {above} cells above threshold")
    export_to_midi(pr, f'Transformer_Generated_Sample_{sample_idx+1}.mid', thresh=0.001)

# Also overwrite the main comparison file with sample 1
import shutil
shutil.copy('Transformer_Generated_Sample_1.mid', 'Transformer_Generated_Sample.mid')
print("\nDone! 10 Transformer MIDI samples exported.")

Sample 1: 64000 cells above threshold
Exported: Transformer_Generated_Sample_1.mid  (128 notes)
Sample 2: 64000 cells above threshold
Exported: Transformer_Generated_Sample_2.mid  (128 notes)
Sample 3: 64000 cells above threshold
Exported: Transformer_Generated_Sample_3.mid  (128 notes)
Sample 4: 64000 cells above threshold
Exported: Transformer_Generated_Sample_4.mid  (128 notes)
Sample 5: 64000 cells above threshold
Exported: Transformer_Generated_Sample_5.mid  (128 notes)
Sample 6: 64000 cells above threshold
Exported: Transformer_Generated_Sample_6.mid  (128 notes)
Sample 7: 64000 cells above threshold
Exported: Transformer_Generated_Sample_7.mid  (128 notes)
Sample 8: 64000 cells above threshold
Exported: Transformer_Generated_Sample_8.mid  (128 notes)
Sample 9: 64000 cells above threshold
Exported: Transformer_Generated_Sample_9.mid  (128 notes)
Sample 10: 64000 cells above threshold
Exported: Transformer_Generated_Sample_10.mid  (128 notes)

Done! 10 Transformer MIDI samples exp

In [8]:
import torch
import numpy as np
import pretty_midi
import shutil

def binarize_topk(pr, k=3):
    """Keep only top-k pitches per time step."""
    result = np.zeros_like(pr)
    for t in range(pr.shape[1]):
        col = pr[:, t]
        if col.max() < 1e-6:
            continue
        top_indices = np.argsort(col)[-k:]
        result[top_indices, t] = 1.0
    return result

def export_to_midi(pianoroll, filename, fs=16):
    pm = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0)
    note_count = 0
    for note_num in range(128):
        active = pianoroll[note_num, :]
        changes = np.diff(active.astype(int), prepend=0, append=0)
        starts = np.where(changes ==  1)[0]
        ends   = np.where(changes == -1)[0]
        for s, e in zip(starts, ends):
            note = pretty_midi.Note(
                velocity=80, pitch=note_num,
                start=s/fs, end=e/fs
            )
            piano.notes.append(note)
            note_count += 1
    pm.instruments.append(piano)
    pm.write(filename)
    return note_count

# ── Generate 10 samples ───────────────────────────────────────────────────
model_trans.eval()

for sample_idx in range(10):
    seed_idx = sample_idx * 5
    current_sequence = X_tensor[seed_idx:seed_idx+1]
    generated = []

    with torch.no_grad():
        for _ in range(500):
            logits = model_trans(current_sequence)
            prediction = torch.sigmoid(logits)
            generated.append(prediction.cpu().numpy())
            new_step = prediction.unsqueeze(1)
            current_sequence = torch.cat(
                (current_sequence[:, 1:, :], new_step), dim=1
            )

    pr = np.array(generated).squeeze().T  # (128, 500)
    binary_pr = binarize_topk(pr, k=3)
    n = export_to_midi(binary_pr, f'Transformer_Generated_Sample_{sample_idx+1}.mid')
    print(f"Transformer Sample {sample_idx+1}: {n} notes written")

shutil.copy('Transformer_Generated_Sample_1.mid', 'Transformer_Generated_Sample.mid')
print("\nDone! 10 Transformer MIDI samples exported.")

Transformer Sample 1: 11 notes written
Transformer Sample 2: 8 notes written
Transformer Sample 3: 9 notes written
Transformer Sample 4: 8 notes written
Transformer Sample 5: 8 notes written
Transformer Sample 6: 8 notes written
Transformer Sample 7: 8 notes written
Transformer Sample 8: 8 notes written
Transformer Sample 9: 9 notes written
Transformer Sample 10: 10 notes written

Done! 10 Transformer MIDI samples exported.
